# 아키텍처별 런타임 평가

**서비스별 TP / FP / TN / FN 평가**를 학생 아키텍처 5종
(1x8 / 2x8 / 1x16 / 2x16 / 2x32)에 대해 각각 수행하고 나란히 비교한다.

## 읽는 구조

```
<test 폴더>/
  models_1x8/  auth/ post/ comment/ frontend/   각 {student_ts.pt, ocsvm.pkl, threshold.json}
  models_2x8/  …
  models_1x16/ …
  models_2x16/ …
  models_2x32/ …
  auth/auth_converter.py   post/…  comment/…  frontend/…
  converter.py  converter_common.py  detector.py  handler.py
  test_pcap/benign/*.pcap  test_pcap/attack/*.pcap
```

학습 노트북이 만든 `models_<arch>/<svc>/` 구조를 **그대로** 읽는다. 배포 규약인
`<svc>/<svc>_model/` 로 옮기거나 별도 디렉터리를 만들 필요가 없다.
`models_*` 디렉터리는 자동 탐지하므로, 일부 arch 만 있어도 있는 것만 돌아간다.

## 평가 방식 (`run_pipeline.ipynb` 와 동일)

- 세션 단위 슬라이딩 윈도우(5, stride=1)
- attack image = benign 4 + attack 1 (혼합 정책)
- mysql 제외, `post`/`comment` 의 `d1` 제외
- 판정: `ocsvm.decision_function(embed) < threshold_df` → anomaly

## 재보정 실패(ROLLBACK) 서비스

학습 때 FPR cap 안에서 REQUIRED recall 을 못 맞춘 서비스는 `threshold.json` 이
**학습 시점 값**으로 남아 있다. 이 노트북은 그런 서비스도 평가하되 `ROLLBK` 로 표시한다.

## §1. 설정 — `models_*` 탐지 및 구조 검증

`models_<arch>/` 를 자동 탐지하고, arch × 서비스마다 `student_ts.pt` / `ocsvm.pkl` /
`threshold.json` 이 있는지 먼저 확인한다. 동시에 각 서비스의 재보정 여부(`recalibrated`)와
임계값을 표로 제시

In [ ]:
import os, sys, json, collections, warnings, re
from pathlib import Path

warnings.filterwarnings("ignore")
import numpy as np

# ── 루트 = 이 노트북이 있는 디렉터리(test 폴더) ─────────────────────────────
ROOT = Path.cwd()

SERVICES = ["auth", "post", "comment", "frontend"]
REMOVED_SERVICES = ["mysql"]

# ── 경로 규약 (스윕) ────────────────────────────────────────────────────────
#   <root>/models_<arch>/<svc>/{student_ts.pt, ocsvm.pkl, threshold.json, ...}
#   <root>/<svc>/<svc>_converter.py          (컨버터는 기존 배포 구조 유지)
#   <root>/test_pcap/{benign,attack}/*.pcap
PCAP_ROOT = ROOT / "test_pcap_notrecal"

def arch_dir(arch):        return ROOT / f"models_{arch}"
def arch_svc_dir(arch, s): return ROOT / f"models_{arch}" / s

# ── models_* 자동 탐지 (있는 것만 스윕) ─────────────────────────────────────
_ORDER = ["1x8", "2x8", "1x16", "2x16", "2x32"]
_found = {p.name[len("models_"):] for p in ROOT.glob("models_*") if p.is_dir()}
ARCHS = [a for a in _ORDER if a in _found] + sorted(_found - set(_ORDER))

if not ARCHS:
    raise FileNotFoundError(f"models_<arch>/ 디렉터리가 없습니다: {ROOT}")

# ── 구조 검증 + 서비스별 재보정 여부 수집 ───────────────────────────────────
_problems = []
ARCH_STATUS = {}          # arch -> svc -> {"recalibrated": bool, "thr": float, ...}

for _svc in SERVICES:
    if not (ROOT / _svc / f"{_svc}_converter.py").is_file():
        _problems.append(f"컨버터 없음: {_svc}/{_svc}_converter.py")
for _f in ("converter.py", "converter_common.py", "detector.py", "handler.py"):
    if not (ROOT / _f).is_file():
        _problems.append(f"루트 공유 모듈 없음: {_f}")
for _d in ("benign", "attack"):
    if not (PCAP_ROOT / _d).is_dir():
        _problems.append(f"pcap 디렉터리 없음: test_pcap_notrecal/{_d}/")

for _a in ARCHS:
    ARCH_STATUS[_a] = {}
    for _svc in SERVICES:
        d = arch_svc_dir(_a, _svc)
        if not d.is_dir():
            _problems.append(f"모델 디렉터리 없음: models_{_a}/{_svc}/"); continue
        if not (d / "student_ts.pt").is_file():
            _problems.append(f"인코더 없음: models_{_a}/{_svc}/student_ts.pt")
        for _f in ("ocsvm.pkl", "threshold.json"):
            if not (d / _f).is_file():
                _problems.append(f"모델 파일 없음: models_{_a}/{_svc}/{_f}")
        if (d / "threshold.json").is_file():
            m = json.loads((d / "threshold.json").read_text(encoding="utf-8"))
            ARCH_STATUS[_a][_svc] = {
                "recalibrated": bool(m.get("recalibrated")),
                "thr": m.get("threshold_df"),
                "arch": m.get("arch"),
                "vec_len": m.get("vec_len"),
            }

if _problems:
    raise FileNotFoundError("구조 문제:\n  - " + "\n  - ".join(_problems))

print("ROOT      =", ROOT)
print("PCAP_ROOT =", PCAP_ROOT)
print("SERVICES  =", SERVICES, f"(제거됨: {REMOVED_SERVICES})")
print("ARCHS     =", ARCHS)

print(f"\n{'arch':8s}" + "".join(f"{s:>14s}" for s in SERVICES))
print("-" * (8 + 14 * len(SERVICES)))
for _a in ARCHS:
    line = f"{_a:8s}"
    for _svc in SERVICES:
        st = ARCH_STATUS[_a][_svc]
        line += f"{('OK' if st['recalibrated'] else 'ROLLBK'):>7s}{st['thr']:7.1f}"
    print(line)
print("\nOK=재보정본 / ROLLBK=학습 시점 임계값(운영 성능 아님, 참고용으로만 비교)")


ROOT      = c:\test
PCAP_ROOT = c:\test\test_pcap_notrecal
SERVICES  = ['auth', 'post', 'comment', 'frontend'] (제거됨: ['mysql'])
ARCHS     = ['1x8', '2x8', '1x16', '2x16', '2x32']

arch              auth          post       comment      frontend
----------------------------------------------------------------
1x8      ROLLBK   -1.2     OK   -0.0 ROLLBK   -2.2 ROLLBK   -0.5
2x8          OK  -43.2     OK  -14.0 ROLLBK -111.0     OK   -0.1
1x16     ROLLBK  -29.2     OK  -18.2 ROLLBK   -5.9 ROLLBK  -37.3
2x16         OK  -44.6     OK   -7.6     OK   -0.2     OK   -0.0
2x32     ROLLBK  -60.5     OK   -1.8     OK   -0.1 ROLLBK -296.0

OK=재보정본 / ROLLBK=학습 시점 임계값(운영 성능 아님, 참고용으로만 비교)


## §2. 모듈 임포트 + 스윕용 Detector 래퍼

컨버터·핸들러는 배포 구조 그대로 쓰고, **모델 경로만** 스윕 규약으로 바꾼다.

`detector.Detector` 의 기본 규약은 `<root>/<svc>/<svc>_model/` 인데 학습 산출물은
`models_<arch>/<svc>/` 다. 그래서 `model_dir` 을 직접 넘기는 얇은 래퍼 `sweep_detector` 를
정의한다. **`detector.py` 자체는 수정하지 않는다.**

마지막에 첫 arch/첫 서비스로 실제 로드를 한 번 해본다. 5종을 다 돌리고 나서 경로 문제를
발견하는 것보다 낫다.

In [ ]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import converter as _conv_mod
from converter import (
    Converter, read_pcap, Packet, WIN_SIZE, FEAT_LEN, SVC_KIND,
    ACTIVE_SERVICES, MODULE_FILES,
)
from handler import Handler, ACT_FORWARD, ACT_RECORD_REQUEST, ACT_TRIGGER_RELAY
from detector import Detector as _BaseDetector    # 원본 클래스(배포 규약)


# ── 스윕용 Detector 래퍼 ────────────────────────────────────────────────────
# 배포 규약은 <root>/<svc>/<svc>_model/ 이지만, 학습 산출물은 models_<arch>/<svc>/ 다.
# 평가 셀 본문(verbatim)이 `Detector(svc, models_root=EVAL_MODELS_ROOT)` 로 호출하므로
# 그 호출을 가로챌 래퍼가 필요하다.
#
#   전역 이름 `Detector` 를 가리지 않는 대신 별도 이름으로 두고, run_eval() 안에서 **지역 변수로** 바인딩한다.
#   지역 바인딩은 어떤 셀이 전역을 건드리든 영향받지 않는다.
def sweep_detector(service, models_root=None, device="cpu", threshold_key="threshold_df"):
    """models_root/<svc>/ 를 모델 디렉터리로 사용한다(스윕 규약)."""
    if models_root is None:
        raise ValueError("models_root 필요 (models_<arch> 경로)")
    return _BaseDetector(service, model_dir=os.path.join(str(models_root), service),
                         device=device, threshold_key=threshold_key)


sweep_detector.__sweep_wrapper__ = True


print("converter 파사드 :", _conv_mod.__file__)
print("FEAT_LEN / WIN_SIZE :", FEAT_LEN, "/", WIN_SIZE)
for _svc in SERVICES:
    _cls = _conv_mod.REGISTRY[_svc]
    print(f"  {_svc:9s} {_cls.__name__:18s} kind={_cls.KIND:9s} {MODULE_FILES[_svc]}")
assert set(ACTIVE_SERVICES) == set(SERVICES), (ACTIVE_SERVICES, SERVICES)

# 래퍼가 실제로 동작하는지 첫 arch/첫 서비스로 즉시 확인
_probe = sweep_detector(SERVICES[0], models_root=str(arch_dir(ARCHS[0])))
print(f"\nDetector 확인: {ARCHS[0]}/{SERVICES[0]} "
      f"thr={_probe.threshold:+.4f}  encoder={_probe.encoder_source}")
del _probe


converter 파사드 : c:\test\converter.py
FEAT_LEN / WIN_SIZE : 20 / 5
  auth      AuthConverter      kind=flow      c:\test\auth\auth_converter.py
  post      PostConverter      kind=backend   c:\test\post\post_converter.py
  comment   CommentConverter   kind=backend   c:\test\comment\comment_converter.py
  frontend  FrontendConverter  kind=frontend  c:\test\frontend\frontend_converter.py

Detector 확인: 1x8/auth thr=-1.1650  encoder=TorchScript(student_ts.pt)


## §3. 평가 설정 — PCAP 전수 인식 + 제외 정책

`run_pipeline.ipynb` 와 동일하다. `test_pcap/` 실물이 평가 대상을 결정하고,
mysql(서비스 제거됨)과 `post`/`comment` 의 `d1`(탐지 범위 밖)은 인식하되 제외한다.

모델 경로(`EVAL_MODELS_ROOT`)만 여기서 정하지 않는다 — arch 마다 다르므로
`run_eval(arch)` 안에서 `models_<arch>` 로 지정된다.

In [ ]:
from pathlib import Path
from collections import defaultdict

import numpy as np
from sklearn.metrics import roc_auc_score

from converter import Converter, read_pcap, WIN_SIZE
from detector import Detector

# ── 경로/서비스: 셀 2의 전역값을 그대로 사용(추정·환경변수 폴백 없음) ────────
EVAL_TEST_PCAP   = PCAP_ROOT
# ★ 스윕: 모델 루트는 arch 마다 다르다(models_<arch>/). run_eval(arch) 안에서 덮어쓴다.
EVAL_MODELS_ROOT = None
EVAL_SERVICES    = list(SERVICES)                     # 평가 대상 (mysql 제외됨)

# pcap 파일명 해석에 쓰는 서비스 집합.
#   ★ 제거된 mysql 도 여기 포함시킨다. 그래야 test_pcap 에 남아 있는 mysql 트래픽을
#     "정체불명 파일"이 아니라 "인식했지만 의도적으로 제외한 파일"로 처리할 수 있다.
KNOWN_SERVICES = list(SERVICES) + list(REMOVED_SERVICES)

# ── 평가 제외 정책 ──────────────────────────────────────────────────────────
#   1) mysql : 서비스 디렉터리(모델·컨버터)가 제거됨 → 평가 불가. pcap 은 존재해도 무시.
#   2) post/comment 의 d1 : 탐지 범위 밖(정책). auth 등 다른 서비스의 d1 은 유지.
EXCLUDED_SERVICES = set(REMOVED_SERVICES)
EXCLUDED_ATTACK_SCENARIOS = {
    ("post", "d1"),
    ("comment", "d1"),
}

# ── 참조용 시나리오 목록(실제 평가 대상은 아래 디렉터리 스캔이 결정) ────────
EVAL_SCENS = {
    "auth":     ["cred_enum", "k1", "k2"],
    "post":     ["enum_seq", "l3", "l2", "k1", "k2"],
    "comment":  ["enum_seq", "l3", "l2", "k1", "k2"],
    "frontend": ["scan_seq", "r1", "k1", "k2"],
}

# ------------------------------------------------------------
# mixed attack 정책
#   WIN_SIZE = 5 이면  B B B B A1 B B B B A2 ...
#   stride=1 이므로 attack packet 1개는 연속 WIN_SIZE 개의 image 에 영향을 준다.
# ------------------------------------------------------------
EVAL_BENIGN_GAP = WIN_SIZE - 1


def _parse_attack_pcap(path: Path):
    stem = path.stem
    for svc in KNOWN_SERVICES:
        prefix = f"attack_{svc}_"
        if stem.startswith(prefix):
            scen = stem[len(prefix):]
            if scen:
                return svc, scen
    raise ValueError(f"공격 PCAP 이름을 해석할 수 없음: {path.name}")


def _parse_benign_pcap(path: Path):
    stem = path.stem
    for svc in KNOWN_SERVICES:
        if stem == f"benign_{svc}":
            return svc
    raise ValueError(f"benign PCAP 이름을 해석할 수 없음: {path.name}")


if not EVAL_TEST_PCAP.exists():
    raise FileNotFoundError(f"test_pcap 폴더 없음: {EVAL_TEST_PCAP.resolve()}")

# ------------------------------------------------------------
# 1) 디렉터리 실물을 기준으로 모든 PCAP 수집(= 인식)
# ------------------------------------------------------------
_eval_attack_paths = sorted((EVAL_TEST_PCAP / "attack").glob("*.pcap"))
_eval_benign_paths = sorted((EVAL_TEST_PCAP / "benign").glob("*.pcap"))

_all_attack = [(*_parse_attack_pcap(p), p) for p in _eval_attack_paths]
_all_benign = [(_parse_benign_pcap(p), p) for p in _eval_benign_paths]

# ------------------------------------------------------------
# 2) 제외 정책 적용(= 인식했지만 평가하지 않음)
# ------------------------------------------------------------
EVAL_ATTACK_FILES = []
_skipped_attack = []

for svc, scen, p in _all_attack:
    if svc in EXCLUDED_SERVICES:
        _skipped_attack.append((svc, scen, p, "서비스 제거됨"))
    elif (svc, scen) in EXCLUDED_ATTACK_SCENARIOS:
        _skipped_attack.append((svc, scen, p, "탐지 범위 밖(정책)"))
    else:
        EVAL_ATTACK_FILES.append((svc, scen, p))

EVAL_BENIGN_FILES = defaultdict(list)
_skipped_benign = []

for svc, p in _all_benign:
    if svc in EXCLUDED_SERVICES:
        _skipped_benign.append((svc, p, "서비스 제거됨"))
    else:
        EVAL_BENIGN_FILES[svc].append(p)

# ------------------------------------------------------------
# 3) 검증
# ------------------------------------------------------------
# 3-a) EVAL_SCENS 에 적힌 시나리오의 pcap 이 실제로 있는지(제외분은 면제)
_expected = {(svc, scen) for svc, scens in EVAL_SCENS.items() for scen in scens
             if svc not in EXCLUDED_SERVICES and (svc, scen) not in EXCLUDED_ATTACK_SCENARIOS}
_actual = {(svc, scen) for svc, scen, _ in EVAL_ATTACK_FILES}
_missing = sorted(_expected - _actual)
if _missing:
    raise FileNotFoundError(f"EVAL_SCENS 에 정의됐지만 실제 PCAP 이 없는 항목: {_missing}")

# 3-b) 평가 대상 서비스에 benign pcap 이 있는지
_missing_benign = [svc for svc in EVAL_SERVICES if not EVAL_BENIGN_FILES.get(svc)]
if _missing_benign:
    raise FileNotFoundError(f"benign PCAP 이 없는 서비스: {_missing_benign}")

# 3-c) test_pcap 아래 pcap 중 '해석조차 안 된' 파일이 없는지
#      (제외된 mysql 파일은 위에서 인식됐으므로 여기서 걸리지 않는다)
_all_pcaps   = {p.resolve() for p in EVAL_TEST_PCAP.rglob("*.pcap")}
_recognized  = {p.resolve() for _, _, p in _all_attack}
_recognized |= {p.resolve() for _, p in _all_benign}
_unhandled = sorted(str(p) for p in (_all_pcaps - _recognized))
if _unhandled:
    raise RuntimeError(f"이름을 해석할 수 없는 PCAP 이 있음: {_unhandled}")

# ------------------------------------------------------------
# 4) 요약 출력
# ------------------------------------------------------------
print(f"TEST_PCAP   = {EVAL_TEST_PCAP.resolve()}")
print(f"인식한 PCAP = {len(_all_pcaps)}개 "
      f"(benign={len(_eval_benign_paths)}, attack={len(_eval_attack_paths)})")
print(f"혼합 정책   = window {WIN_SIZE}개 중 benign {EVAL_BENIGN_GAP} + attack 1, stride=1")

print("\n[평가 제외]")
for svc, scen, p, why in _skipped_attack:
    print(f"  attack  {svc}/{scen:10s} {p.name:34s} — {why}")
for svc, p, why in _skipped_benign:
    print(f"  benign  {svc:17s} {p.name:34s} — {why}")
if not _skipped_attack and not _skipped_benign:
    print("  (없음)")

print("\n[실제 평가 대상]")
for svc in EVAL_SERVICES:
    scens = [scen for s, scen, _ in EVAL_ATTACK_FILES if s == svc]
    print(f"  {svc:9s}: benign {len(EVAL_BENIGN_FILES[svc])}개 / "
          f"attack {len(scens)}개 — {', '.join(scens)}")


TEST_PCAP   = C:\test\test_pcap_notrecal
인식한 PCAP = 29개 (benign=5, attack=24)
혼합 정책   = window 5개 중 benign 4 + attack 1, stride=1

[평가 제외]
  attack  comment/d1         attack_comment_d1.pcap             — 탐지 범위 밖(정책)
  attack  mysql/c2         attack_mysql_c2.pcap               — 서비스 제거됨
  attack  mysql/e1         attack_mysql_e1.pcap               — 서비스 제거됨
  attack  mysql/k1         attack_mysql_k1.pcap               — 서비스 제거됨
  attack  mysql/k2         attack_mysql_k2.pcap               — 서비스 제거됨
  attack  post/d1         attack_post_d1.pcap                — 탐지 범위 밖(정책)
  benign  mysql             benign_mysql.pcap                  — 서비스 제거됨

[실제 평가 대상]
  auth     : benign 1개 / attack 4개 — cred_enum, d1, k1, k2
  post     : benign 1개 / attack 5개 — enum_seq, k1, k2, l2, l3
  comment  : benign 1개 / attack 5개 — enum_seq, k1, k2, l2, l3
  frontend : benign 1개 / attack 4개 — k1, k2, r1, scan_seq


## §4. 평가 함수 정의

세션 단위 벡터 추출, 순수 benign 윈도우, mixed attack 윈도우(benign 4 + attack 1) 구성과
점수화 함수들. `run_pipeline.ipynb` 에서 **그대로** 가져왔다. 실행은 하지 않는다.

In [ ]:
# ===== [NEW CELL 2 교체본] =====
# 세션별 벡터화 + pure benign / runtime-style mixed attack 이미지 생성
#
# 정책
#   1) benign:
#        실제 benign session 내부에서만 stride=1 sliding window
#
#   2) 일반 attack:
#        BBBB + A1 A2 A3 ... An + BBBB
#
#        → 공격 진입:
#          BBBBA     (attack 1/5)
#          BBBAA     (attack 2/5)
#          BBAAA     (attack 3/5)
#          BAAAA     (attack 4/5)
#          AAAAA     (attack 5/5)
#
#        → 이후 실제 attack session의 sequential 구조를 그대로 유지
#
#   3) k1 / k2:
#        공격 packet 하나만 포함돼도 anomaly image로 정의하는 기존 정책 유지
#
#        각 attack packet마다 독립적으로
#
#          BBBB A BBBB
#
#        를 구성
#
#        → 만들어지는 positive image 5개 모두 attack occupancy = 1/5
#
# 랜덤 선택 없음.
# attack session 내부 순서 및 benign source session 내부 순서 유지.

from collections import defaultdict, deque
from pathlib import Path

import numpy as np


# ------------------------------------------------------------
# 정책:
# 이 시나리오들은 공격 패킷 하나만 window에 포함돼도 true attack image로 평가한다.
# ------------------------------------------------------------

SINGLE_PACKET_ATTACK_SCENS = {
    "k1",
    "k2",
}


def _scenario_from_attack_source(source_name: str, svc: str):
    """
    attack_post_k1.pcap -> k1
    attack_auth_cred_enum.pcap -> cred_enum
    """

    stem = Path(source_name).stem
    prefix = f"attack_{svc}_"

    if stem.startswith(prefix):
        return stem[len(prefix):]

    # 예상 형식이 아니더라도 일반 sequential 정책으로 처리
    return stem


def _extract_session_vectors(
    svc: str,
    pcap_path: Path,
):
    """
    PCAP 전체를 Converter.extract 경로로 처리한 뒤
    실제 session_id별 feature vector sequence로 묶는다.

    중요:
      - random sampling 없음
      - session 내부 packet 순서 유지
      - Converter.extract가 None인 packet은
        실제 runtime과 동일하게 이미지 대상에서 제외
      - f18/f19 temporal feature는 원래 PCAP/session에서 계산된 값을 유지

    즉 synthetic mixing은 raw packet이 아니라
    '원래 source session에서 정상적으로 만들어진 feature vector'를
    이용해 수행한다.
    """

    conv = Converter(svc)
    conv.reset()

    packets = read_pcap(
        str(pcap_path),
        "auto",
        limit=None,
    )

    sessions = defaultdict(list)
    kept = 0

    for pkt in packets:

        r = conv.extract(pkt)

        if r is None:
            continue

        vec, _is_request = r

        sessions[pkt.session_id].append(
            vec.copy()
        )

        kept += 1

    # dict 삽입 순서 =
    # 해당 session이 PCAP에서 처음 등장한 순서
    ordered = [
        (sid, vecs)
        for sid, vecs in sessions.items()
    ]

    return ordered, {
        "pcap":
            pcap_path.name,

        "raw_egress_packets":
            len(packets),

        "kept_packets":
            kept,

        "sessions":
            len(ordered),
    }


# ============================================================
# PURE BENIGN
# ============================================================

def _score_pure_benign(
    svc: str,
    det,
    benign_sessions,
):
    """
    실제 benign session 내부에서만
    stride=1 sliding window 생성.

        B1 B2 B3 B4 B5
           B2 B3 B4 B5 B6
              B3 B4 B5 B6 B7
              ...

    서로 다른 실제 benign session을 하나의 image에 섞지 않는다.
    """

    image_conv = Converter(svc)

    scores = []
    per_session_images = []

    for source_name, sid, vecs in benign_sessions:

        n_img = max(
            0,
            len(vecs) - WIN_SIZE + 1,
        )

        if n_img == 0:
            continue

        per_session_images.append(
            (
                source_name,
                sid,
                n_img,
            )
        )

        for start in range(n_img):

            window = vecs[
                start:start + WIN_SIZE
            ]

            img = image_conv.to_image(
                window
            )

            scores.append(
                det.score(img)
            )

    return (
        np.asarray(
            scores,
            dtype=np.float64,
        ),
        per_session_images,
    )


# ============================================================
# BENIGN CONTEXT
# ============================================================

def _benign_context(
    vecs,
    start,
    n,
):
    """
    한 benign source session 안에서 n개의 vector를 가져온다.

    충분히 긴 session이면 그대로 연속 구간을 사용한다.
    짧으면 동일 session 내부에서 circular하게 사용한다.

    서로 다른 benign session을 한 context 안에서 섞지는 않는다.
    """

    if not vecs:
        raise ValueError(
            "empty benign session"
        )

    out = []

    pos = start % len(vecs)

    for _ in range(n):

        out.append(
            vecs[pos]
        )

        pos = (
            pos + 1
        ) % len(vecs)

    return out, pos


# ============================================================
# MODE 1
# 일반 sequential attack
# ============================================================

def _score_sequential_attack_session(
    svc,
    det,
    attack_vecs,
    benign_vecs,
    benign_start=0,
):
    """
    일반적인 sequential 공격용.

    하나의 실제 attack session 전체를 끊지 않고:

        BBBB + A1 A2 A3 ... An + BBBB

    로 구성한다.

    WIN_SIZE=5라면:

        BBBBA1       occupancy 1
        BBBA1A2      occupancy 2
        BBA1A2A3     occupancy 3
        BA1A2A3A4    occupancy 4
        A1A2A3A4A5   occupancy 5
        A2A3A4A5A6   occupancy 5
        ...
        A...A B      occupancy 4
        A... B B     occupancy 3
        ...
        A BBBB       occupancy 1

    생성된 모든 image는 attack column >= 1이므로
    true attack image다.
    """

    image_conv = Converter(svc)

    scores = []

    # occupancy별 score
    occupancy_scores = defaultdict(list)

    prefix, pos = _benign_context(
        benign_vecs,
        benign_start,
        WIN_SIZE - 1,
    )

    suffix, pos = _benign_context(
        benign_vecs,
        pos,
        WIN_SIZE - 1,
    )

    # (vector, label)
    # label: benign=0, attack=1
    stream = (
        [(v, 0) for v in prefix]
        + [(v, 1) for v in attack_vecs]
        + [(v, 0) for v in suffix]
    )

    win_vec = deque(
        maxlen=WIN_SIZE
    )

    win_label = deque(
        maxlen=WIN_SIZE
    )

    for vec, label in stream:

        win_vec.append(vec)
        win_label.append(label)

        if len(win_vec) < WIN_SIZE:
            continue

        occupancy = int(
            sum(win_label)
        )

        # attack 평가 구간에는 pure benign image가
        # 생성되면 안 된다.
        if occupancy <= 0:
            raise AssertionError(
                f"{svc}: sequential attack 평가에서 "
                f"pure benign window가 생성됨"
            )

        img = image_conv.to_image(
            list(win_vec)
        )

        score = det.score(img)

        scores.append(score)

        occupancy_scores[
            occupancy
        ].append(score)

    expected = (
        len(attack_vecs)
        + WIN_SIZE - 1
    )

    if len(scores) != expected:

        raise AssertionError(
            f"{svc}: sequential attack image 수 불일치: "
            f"actual={len(scores)}, "
            f"expected={expected}"
        )

    return (
        np.asarray(
            scores,
            dtype=np.float64,
        ),
        {
            k: np.asarray(
                v,
                dtype=np.float64,
            )
            for k, v
            in occupancy_scores.items()
        },
        pos,
    )


# ============================================================
# MODE 2
# k1 / k2 — single packet attack
# ============================================================

def _score_single_packet_attack_session(
    svc,
    det,
    attack_vecs,
    benign_vecs,
    benign_start=0,
):
    """
    k1 / k2 전용.

    공격 packet 하나만 들어와도 anomaly image로 정의하는
    기존 평가 정책을 그대로 유지한다.

    각 attack packet A마다 별도의 평가 sequence:

        B1 B2 B3 B4 A B5 B6 B7 B8

    를 만든다.

    stride=1이면:

        B B B B A
        B B B A B
        B B A B B
        B A B B B
        A B B B B

    총 5개 image가 생성되며
    전부 attack occupancy = 1/5.

    다음 attack packet은 새로운 독립 sequence로 평가하므로
    서로 다른 attack packet이 같은 window에 들어가지 않는다.
    """

    image_conv = Converter(svc)

    scores = []

    occupancy_scores = defaultdict(list)

    pos = benign_start

    for attack_vec in attack_vecs:

        prefix, pos = _benign_context(
            benign_vecs,
            pos,
            WIN_SIZE - 1,
        )

        suffix, pos = _benign_context(
            benign_vecs,
            pos,
            WIN_SIZE - 1,
        )

        stream = (
            [(v, 0) for v in prefix]
            + [(attack_vec, 1)]
            + [(v, 0) for v in suffix]
        )

        win_vec = deque(
            maxlen=WIN_SIZE
        )

        win_label = deque(
            maxlen=WIN_SIZE
        )

        this_attack_images = 0

        for vec, label in stream:

            win_vec.append(vec)
            win_label.append(label)

            if len(win_vec) < WIN_SIZE:
                continue

            occupancy = int(
                sum(win_label)
            )

            # k1/k2는 반드시 attack column이 정확히 하나
            if occupancy != 1:
                raise AssertionError(
                    f"{svc}: k1/k2 window의 "
                    f"attack occupancy={occupancy}, expected=1"
                )

            img = image_conv.to_image(
                list(win_vec)
            )

            score = det.score(img)

            scores.append(score)

            occupancy_scores[1].append(
                score
            )

            this_attack_images += 1

        # attack packet 1개는 stride=1, window=5에서
        # 정확히 5개 image에 존재
        if this_attack_images != WIN_SIZE:

            raise AssertionError(
                f"{svc}: single attack packet으로 "
                f"{this_attack_images}개 image 생성, "
                f"expected={WIN_SIZE}"
            )

    expected = (
        len(attack_vecs)
        * WIN_SIZE
    )

    if len(scores) != expected:

        raise AssertionError(
            f"{svc}: k1/k2 image 수 불일치: "
            f"actual={len(scores)}, "
            f"expected={expected}"
        )

    return (
        np.asarray(
            scores,
            dtype=np.float64,
        ),
        {
            1: np.asarray(
                occupancy_scores[1],
                dtype=np.float64,
            )
        },
        pos,
    )


# ============================================================
# ATTACK 전체 wrapper
# 기존 Cell 3와 동일한 함수 이름/인자/반환 형식 유지
# ============================================================

def _score_mixed_attack(
    svc: str,
    det,
    attack_sessions,
    benign_background_sessions,
):
    if not benign_background_sessions:

        raise RuntimeError(
            f"{svc}: mixed attack용 "
            f"benign background session이 없음"
        )

    scores_all = []

    pair_info = []

    occupancy_scores_all = defaultdict(
        list
    )

    attack_packets_used = 0

    mode_counts = defaultdict(int)

    for atk_idx, (
        attack_source,
        attack_sid,
        attack_vecs,
    ) in enumerate(
        attack_sessions
    ):

        if not attack_vecs:
            continue

        scen = _scenario_from_attack_source(
            attack_source,
            svc,
        )

        # random 아님:
        # attack session 등장 순서대로
        # benign session을 round-robin pairing
        bg_source, bg_sid, bg_vecs = (
            benign_background_sessions[
                atk_idx
                % len(
                    benign_background_sessions
                )
            ]
        )

        if not bg_vecs:

            raise RuntimeError(
                f"{svc}: 빈 benign "
                f"background session 선택됨"
            )

        # session마다 deterministic start offset
        bg_start = (
            atk_idx * WIN_SIZE
        ) % len(bg_vecs)


        # ----------------------------------------------------
        # k1/k2:
        # attack 1개만 들어와도 true attack image
        # ----------------------------------------------------

        if scen in SINGLE_PACKET_ATTACK_SCENS:

            mode = "single-packet"

            (
                session_scores,
                occupancy_scores,
                _,
            ) = _score_single_packet_attack_session(
                svc,
                det,
                attack_vecs,
                bg_vecs,
                bg_start,
            )


        # ----------------------------------------------------
        # 나머지:
        # 실제 attack session 전체를 연속으로 유지
        # ----------------------------------------------------

        else:

            mode = "sequential"

            (
                session_scores,
                occupancy_scores,
                _,
            ) = _score_sequential_attack_session(
                svc,
                det,
                attack_vecs,
                bg_vecs,
                bg_start,
            )


        scores_all.extend(
            session_scores.tolist()
        )

        for occupancy, vals in (
            occupancy_scores.items()
        ):

            occupancy_scores_all[
                occupancy
            ].extend(
                vals.tolist()
            )

        attack_packets_used += len(
            attack_vecs
        )

        mode_counts[mode] += 1


        # session별 occupancy image 개수
        occ_counts = {
            int(k): int(len(v))
            for k, v
            in occupancy_scores.items()
        }


        pair_info.append({
            "scenario":
                scen,

            "mode":
                mode,

            "attack_source":
                attack_source,

            "attack_sid":
                attack_sid,

            "attack_packets":
                len(attack_vecs),

            "benign_source":
                bg_source,

            "benign_sid":
                bg_sid,

            "mixed_images":
                len(session_scores),

            "occupancy_counts":
                occ_counts,
        })


    occupancy_scores_np = {
        int(k): np.asarray(
            v,
            dtype=np.float64,
        )
        for k, v
        in occupancy_scores_all.items()
    }


    return (
        np.asarray(
            scores_all,
            dtype=np.float64,
        ),
        {
            # 기존 Cell 3에서 쓰는 키
            "attack_packets_used":
                attack_packets_used,

            "mixed_images":
                len(scores_all),

            "pairs":
                pair_info,

            # 추가 정보
            "mode_counts":
                dict(mode_counts),

            # occupancy별 실제 score
            # 1 = attack 1/5
            # ...
            # 5 = attack 5/5
            "occupancy_scores":
                occupancy_scores_np,

            "occupancy_counts": {
                k: len(v)
                for k, v
                in occupancy_scores_np.items()
            },
        },
    )


# ============================================================
# METRICS
# 기존 Cell 3 호환 유지
# ============================================================

def _safe_div(a, b):

    return (
        float(a) / float(b)
        if b
        else float("nan")
    )


def _metrics_from_scores(
    benign_scores,
    attack_scores,
    threshold,
):
    """
    OCSVM 판정

        score < threshold
            -> anomaly

    true label

        pure benign image
            -> negative

        attack이 1 column 이상 포함된 image
            -> positive

    단,
        k1/k2:
            occupancy=1만 생성

        나머지:
            occupancy=1~5가 attack 진입/지속/종료 과정에 따라 생성
    """

    benign_pred_anom = (
        benign_scores < threshold
    )

    attack_pred_anom = (
        attack_scores < threshold
    )


    FP = int(
        benign_pred_anom.sum()
    )

    TN = int(
        len(benign_scores) - FP
    )

    TP = int(
        attack_pred_anom.sum()
    )

    FN = int(
        len(attack_scores) - TP
    )


    recall = _safe_div(
        TP,
        TP + FN,
    )

    fpr = _safe_div(
        FP,
        FP + TN,
    )

    precision = _safe_div(
        TP,
        TP + FP,
    )

    accuracy = _safe_div(
        TP + TN,
        TP + TN + FP + FN,
    )


    if (
        np.isfinite(precision)
        and np.isfinite(recall)
        and (precision + recall) > 0
    ):
        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
        )

    else:
        f1 = float("nan")


    if (
        len(benign_scores)
        and len(attack_scores)
    ):

        y_true = np.r_[
            np.zeros(
                len(benign_scores),
                dtype=np.int8,
            ),
            np.ones(
                len(attack_scores),
                dtype=np.int8,
            ),
        ]

        # OCSVM은 낮은 score가 anomaly이므로
        # ROC에서는 부호 반전
        y_score = np.r_[
            -benign_scores,
            -attack_scores,
        ]

        auc = float(
            roc_auc_score(
                y_true,
                y_score,
            )
        )

    else:

        auc = float("nan")


    return {
        "TP": TP,
        "FP": FP,
        "TN": TN,
        "FN": FN,

        "recall": recall,
        "fpr": fpr,
        "precision": precision,
        "accuracy": accuracy,
        "f1": f1,
        "auc": auc,
    }

## §5. arch 실행기 `run_eval(arch)`

`run_pipeline.ipynb` 의 평가 실행 셀 본문을 **한 글자도 바꾸지 않고** 함수로 감쌌다.
바뀐 것은 세 가지뿐이다.

1. 전체를 `run_eval()` 안으로 들여쓰기
2. `EVAL_MODELS_ROOT = models_<arch>` 지정
3. `Detector = sweep_detector` **지역 바인딩** — verbatim 본문의 `Detector(...)` 호출이
   지역 변수를 먼저 찾으므로, §3 이 전역 `Detector` 를 덮어써도 안전하다

결과(`EVAL_RESULTS`)를 반환하므로 arch 별로 모아 비교할 수 있다.

In [5]:
# arch 하나에 대한 전체 평가 실행기.

def run_eval(arch):
    """models_<arch>/ 로 전체 test_pcap 평가 → EVAL_RESULTS 반환."""
    assert arch in ARCHS, f"{arch} 없음 (있는 것: {ARCHS})"

    Detector = sweep_detector
    assert getattr(Detector, "__sweep_wrapper__", False), \
        "sweep_detector 가 래퍼가 아닙니다 — §2 셀을 다시 실행하세요"

    EVAL_MODELS_ROOT = str(arch_dir(arch))

    print("#" * 110)
    print(f"### ARCH = {arch}   (models_root = {EVAL_MODELS_ROOT})")
    rb = [s for s in EVAL_SERVICES if not ARCH_STATUS[arch][s]["recalibrated"]]
    if rb:
        print(f"### ⚠️ 재보정 안 된 서비스: {rb} — 학습 시점 임계값 유지")
    print("#" * 110)

    # ===== 전체 test_pcap 평가 + 서비스별 TP/FP/TN/FN =====

    EVAL_RESULTS = {}

    _processed_benign_files = []
    _processed_attack_files = []


    print(
        "\n"
        + "=" * 110
    )

    print(
        "세션 기반 런타임 평가 시작"
    )

    print(
        f"attack image 정책: "
        f"{EVAL_BENIGN_GAP} benign + 1 attack "
        f"/ window={WIN_SIZE} "
        f"/ stride=1"
    )

    print(
        "=" * 110
    )


    for svc in EVAL_SERVICES:

        print(
            f"\n{'#' * 110}"
        )

        print(
            f"■ SERVICE: {svc}"
        )

        print(
            f"{'#' * 110}"
        )


        det = Detector(
            svc,
            models_root=EVAL_MODELS_ROOT,
        )

        thr = det.threshold


        # ========================================================
        # 1. BENIGN
        #
        # 모든 benign PCAP
        # -> 실제 session별
        # -> pure benign sliding window
        # ========================================================

        benign_sessions = []
        benign_pcap_stats = []


        for bp in EVAL_BENIGN_FILES[svc]:

            sess, stat = (
                _extract_session_vectors(
                    svc,
                    bp,
                )
            )

            benign_pcap_stats.append(
                stat
            )

            _processed_benign_files.append(
                bp.resolve()
            )


            for sid, vecs in sess:

                benign_sessions.append(
                    (
                        bp.name,
                        sid,
                        vecs,
                    )
                )


        # mixed attack의 background로 사용할 benign session
        #
        # 최소한 실제 benign window 하나는 만들 수 있는
        # 길이 >= WIN_SIZE session만 사용
        benign_background = [
            item
            for item in benign_sessions
            if len(item[2]) >= WIN_SIZE
        ]


        if not benign_background:

            raise RuntimeError(
                f"{svc}: "
                f"kept packet이 "
                f"{WIN_SIZE}개 이상인 "
                f"benign session이 없음"
            )


        benign_scores, benign_image_info = (
            _score_pure_benign(
                svc,
                det,
                benign_sessions,
            )
        )


        if len(benign_scores) == 0:

            raise RuntimeError(
                f"{svc}: "
                f"pure benign 평가 이미지가 0개"
            )


        print(
            f"threshold = "
            f"{thr:+.6f}"
        )


        print(
            "\n[benign]"
        )


        for st in benign_pcap_stats:

            print(
                f"  {st['pcap']}: "
                f"raw-egress="
                f"{st['raw_egress_packets']}, "
                f"kept="
                f"{st['kept_packets']}, "
                f"sessions="
                f"{st['sessions']}"
            )


        print(
            f"  pure benign images = "
            f"{len(benign_scores)} "
            f"(실제 benign session 내부에서만 "
            f"stride=1 생성)"
        )


        # ========================================================
        # 2. ATTACK
        #
        # 모든 attack PCAP
        # -> 실제 attack session별
        # -> benign background와 deterministic mixing
        # ========================================================

        service_attack_files = [
            (
                scen,
                ap,
            )
            for attack_svc, scen, ap
            in EVAL_ATTACK_FILES
            if attack_svc == svc
        ]


        per_scenario = {}

        attack_score_parts = []

        total_kept_attack_packets = 0


        print(
            "\n[attack: mixed windows]"
        )

        print(
            f"  {'scenario':16s} "
            f"{'raw':>8s} "
            f"{'kept':>8s} "
            f"{'sess':>6s} "
            f"{'mixed-img':>10s} "
            f"{'TP':>8s} "
            f"{'FN':>8s} "
            f"{'recall':>9s}"
        )

        print(
            "  "
            + "-" * 82
        )


        for scen, ap in service_attack_files:

            atk_sess_raw, atk_stat = (
                _extract_session_vectors(
                    svc,
                    ap,
                )
            )

            _processed_attack_files.append(
                ap.resolve()
            )


            attack_sessions = [
                (
                    ap.name,
                    sid,
                    vecs,
                )
                for sid, vecs in atk_sess_raw
                if len(vecs) > 0
            ]


            attack_scores, mix_stat = (
                _score_mixed_attack(
                    svc,
                    det,
                    attack_sessions,
                    benign_background,
                )
            )


            total_kept_attack_packets += (
                mix_stat[
                    "attack_packets_used"
                ]
            )


            # 이 PCAP에서 생성된 mixed image는
            # 모두 true attack
            TP_scen = int(
                (
                    attack_scores
                    < thr
                ).sum()
            )

            FN_scen = int(
                len(attack_scores)
                - TP_scen
            )


            recall_scen = _safe_div(
                TP_scen,
                TP_scen + FN_scen,
            )


            per_scenario[scen] = {

                "pcap":
                    ap.name,

                "scores":
                    attack_scores,

                "TP":
                    TP_scen,

                "FN":
                    FN_scen,

                "recall":
                    recall_scen,

                "raw_egress_packets":
                    atk_stat[
                        "raw_egress_packets"
                    ],

                "kept_attack_packets":
                    mix_stat[
                        "attack_packets_used"
                    ],

                "source_sessions":
                    len(attack_sessions),

                "mixed_images":
                    mix_stat[
                        "mixed_images"
                    ],

                "pairs":
                    mix_stat[
                        "pairs"
                    ],
            }


            if len(attack_scores):

                attack_score_parts.append(
                    attack_scores
                )


            print(
                f"  {scen:16s} "
                f"{atk_stat['raw_egress_packets']:8d} "
                f"{mix_stat['attack_packets_used']:8d} "
                f"{len(attack_sessions):6d} "
                f"{len(attack_scores):10d} "
                f"{TP_scen:8d} "
                f"{FN_scen:8d} "
                f"{recall_scen * 100:8.2f}%"
            )


        if attack_score_parts:

            attack_scores_all = (
                np.concatenate(
                    attack_score_parts
                )
            )

        else:

            attack_scores_all = (
                np.asarray(
                    [],
                    dtype=np.float64,
                )
            )


        # ========================================================
        # 3. SERVICE CONFUSION MATRIX
        # ========================================================

        m = _metrics_from_scores(
            benign_scores,
            attack_scores_all,
            thr,
        )


        print(
            "\n[confusion matrix]"
        )

        print(
            "                         "
            "predicted benign     "
            "predicted anomaly"
        )

        print(
            f"  true benign              "
            f"TN={m['TN']:8d}          "
            f"FP={m['FP']:8d}"
        )

        print(
            f"  true attack              "
            f"FN={m['FN']:8d}          "
            f"TP={m['TP']:8d}"
        )


        print(
            "\n[metrics]"
        )

        print(
            f"  benign images : "
            f"{len(benign_scores)}"
        )

        print(
            f"  attack images : "
            f"{len(attack_scores_all)} "
            f"(전부 benign "
            f"{EVAL_BENIGN_GAP} + attack 1)"
        )

        print(
            f"  attack packets: "
            f"{total_kept_attack_packets} "
            f"(image화 대상 attack packet 전부 사용)"
        )

        print(
            f"  Recall / TPR  : "
            f"{m['recall'] * 100:8.2f}%"
        )

        print(
            f"  FPR           : "
            f"{m['fpr'] * 100:8.2f}%"
        )

        print(
            f"  Precision     : "
            f"{m['precision'] * 100:8.2f}%"
        )

        print(
            f"  Accuracy      : "
            f"{m['accuracy'] * 100:8.2f}%"
        )

        print(
            f"  F1            : "
            f"{m['f1']:8.4f}"
        )

        print(
            f"  ROC-AUC       : "
            f"{m['auc']:8.4f}"
        )


        EVAL_RESULTS[svc] = {

            "threshold":
                thr,

            "benign_scores":
                benign_scores,

            "attack_scores":
                attack_scores_all,

            "per_scenario":
                per_scenario,

            "metrics":
                m,

            "benign_pcap_stats":
                benign_pcap_stats,

            "benign_image_info":
                benign_image_info,

            "kept_attack_packets":
                total_kept_attack_packets,
        }


    # ============================================================
    # 4. 모든 PCAP 파일이 실제 평가되었는지 최종 검증
    # ============================================================

    _expected_benign_paths = {

        p.resolve()

        for paths
        in EVAL_BENIGN_FILES.values()

        for p
        in paths
    }


    _expected_attack_paths = {

        p.resolve()

        for _, _, p
        in EVAL_ATTACK_FILES
    }


    if (
        set(_processed_benign_files)
        != _expected_benign_paths
    ):

        missing = (
            _expected_benign_paths
            - set(_processed_benign_files)
        )

        raise AssertionError(
            "평가되지 않은 benign PCAP: "
            f"{sorted(map(str, missing))}"
        )


    if (
        set(_processed_attack_files)
        != _expected_attack_paths
    ):

        missing = (
            _expected_attack_paths
            - set(_processed_attack_files)
        )

        raise AssertionError(
            "평가되지 않은 attack PCAP: "
            f"{sorted(map(str, missing))}"
        )


    # ============================================================
    # 5. 최종 서비스별 요약
    # ============================================================

    print(
        "\n"
        + "=" * 110
    )

    print(
        "서비스별 최종 요약"
    )

    print(
        "=" * 110
    )


    print(
        f"{'svc':10s} "
        f"{'benign-img':>11s} "
        f"{'attack-img':>11s} "
        f"{'TP':>8s} "
        f"{'FP':>8s} "
        f"{'TN':>8s} "
        f"{'FN':>8s} "
        f"{'Recall':>9s} "
        f"{'FPR':>9s} "
        f"{'AUC':>8s}"
    )

    print(
        "-" * 110
    )


    for svc in EVAL_SERVICES:

        r = EVAL_RESULTS[svc]
        m = r["metrics"]

        print(
            f"{svc:10s} "
            f"{len(r['benign_scores']):11d} "
            f"{len(r['attack_scores']):11d} "
            f"{m['TP']:8d} "
            f"{m['FP']:8d} "
            f"{m['TN']:8d} "
            f"{m['FN']:8d} "
            f"{m['recall'] * 100:8.2f}% "
            f"{m['fpr'] * 100:8.2f}% "
            f"{m['auc']:8.4f}"
        )


    overall_TP = sum(
        r["metrics"]["TP"]
        for r in EVAL_RESULTS.values()
    )

    overall_FP = sum(
        r["metrics"]["FP"]
        for r in EVAL_RESULTS.values()
    )

    overall_TN = sum(
        r["metrics"]["TN"]
        for r in EVAL_RESULTS.values()
    )

    overall_FN = sum(
        r["metrics"]["FN"]
        for r in EVAL_RESULTS.values()
    )


    print(
        "-" * 110
    )

    print(
        "OVERALL confusion counts: "
        f"TP={overall_TP}, "
        f"FP={overall_FP}, "
        f"TN={overall_TN}, "
        f"FN={overall_FN}"
    )

    print(
        "PCAP 전수 사용 확인 완료: "
        f"benign={len(_processed_benign_files)}개, "
        f"attack={len(_processed_attack_files)}개, "
        f"total="
        f"{len(_processed_benign_files) + len(_processed_attack_files)}개"
    )

    return EVAL_RESULTS


SWEEP = {}          # arch -> EVAL_RESULTS

### §6-1. `1x8` 평가 (~1.23K)

`models_1x8/` 로 전체 `test_pcap` 을 평가하고 서비스별 TP / FP / TN / FN 을 낸다.
결과는 `SWEEP["1x8"]` 에 저장되어 §7 비교표에 쓰인다.

In [6]:
if "1x8" in ARCHS:
    SWEEP["1x8"] = run_eval("1x8")
else:
    print("models_1x8/ 없음 — 건너뜀")

##############################################################################################################
### ARCH = 1x8   (models_root = c:\test\models_1x8)
### ⚠️ 재보정 안 된 서비스: ['auth', 'comment', 'frontend'] — 학습 시점 임계값 유지
##############################################################################################################

세션 기반 런타임 평가 시작
attack image 정책: 4 benign + 1 attack / window=5 / stride=1

##############################################################################################################
■ SERVICE: auth
##############################################################################################################
threshold = -1.165003

[benign]
  benign_auth.pcap: raw-egress=109831, kept=109831, sessions=888
  pure benign images = 106282 (실제 benign session 내부에서만 stride=1 생성)

[attack: mixed windows]
  scenario              raw     kept   sess  mixed-img       TP       FN    recall
  ---------------------------------------------------------------------

### §6-2. `2x8` 평가 (~5.69K)

`models_2x8/` 로 전체 `test_pcap` 을 평가하고 서비스별 TP / FP / TN / FN 을 낸다.
결과는 `SWEEP["2x8"]` 에 저장되어 §7 비교표에 쓰인다.

In [7]:
if "2x8" in ARCHS:
    SWEEP["2x8"] = run_eval("2x8")
else:
    print("models_2x8/ 없음 — 건너뜀")

##############################################################################################################
### ARCH = 2x8   (models_root = c:\test\models_2x8)
### ⚠️ 재보정 안 된 서비스: ['comment'] — 학습 시점 임계값 유지
##############################################################################################################

세션 기반 런타임 평가 시작
attack image 정책: 4 benign + 1 attack / window=5 / stride=1

##############################################################################################################
■ SERVICE: auth
##############################################################################################################
threshold = -43.178968

[benign]
  benign_auth.pcap: raw-egress=109831, kept=109831, sessions=888
  pure benign images = 106282 (실제 benign session 내부에서만 stride=1 생성)

[attack: mixed windows]
  scenario              raw     kept   sess  mixed-img       TP       FN    recall
  ----------------------------------------------------------------------------------
  cre

### §6-3. `1x16` 평가 (~12.64K)

`models_1x16/` 로 전체 `test_pcap` 을 평가하고 서비스별 TP / FP / TN / FN 을 낸다.
결과는 `SWEEP["1x16"]` 에 저장되어 §7 비교표에 쓰인다.

In [8]:
if "1x16" in ARCHS:
    SWEEP["1x16"] = run_eval("1x16")
else:
    print("models_1x16/ 없음 — 건너뜀")

##############################################################################################################
### ARCH = 1x16   (models_root = c:\test\models_1x16)
### ⚠️ 재보정 안 된 서비스: ['auth', 'comment', 'frontend'] — 학습 시점 임계값 유지
##############################################################################################################

세션 기반 런타임 평가 시작
attack image 정책: 4 benign + 1 attack / window=5 / stride=1

##############################################################################################################
■ SERVICE: auth
##############################################################################################################
threshold = -29.223119

[benign]
  benign_auth.pcap: raw-egress=109831, kept=109831, sessions=888
  pure benign images = 106282 (실제 benign session 내부에서만 stride=1 생성)

[attack: mixed windows]
  scenario              raw     kept   sess  mixed-img       TP       FN    recall
  ------------------------------------------------------------------

### §6-4. `2x16` 평가 (~13.87K)

`models_2x16/` 로 전체 `test_pcap` 을 평가하고 서비스별 TP / FP / TN / FN 을 낸다.
결과는 `SWEEP["2x16"]` 에 저장되어 §7 비교표에 쓰인다.

In [9]:
if "2x16" in ARCHS:
    SWEEP["2x16"] = run_eval("2x16")
else:
    print("models_2x16/ 없음 — 건너뜀")

##############################################################################################################
### ARCH = 2x16   (models_root = c:\test\models_2x16)
##############################################################################################################

세션 기반 런타임 평가 시작
attack image 정책: 4 benign + 1 attack / window=5 / stride=1

##############################################################################################################
■ SERVICE: auth
##############################################################################################################
threshold = -44.646406

[benign]
  benign_auth.pcap: raw-egress=109831, kept=109831, sessions=888
  pure benign images = 106282 (실제 benign session 내부에서만 stride=1 생성)

[attack: mixed windows]
  scenario              raw     kept   sess  mixed-img       TP       FN    recall
  ----------------------------------------------------------------------------------
  cred_enum            1150     1150     74       

### §6-5. `2x32` 평가 (~87.26K)

`models_2x32/` 로 전체 `test_pcap` 을 평가하고 서비스별 TP / FP / TN / FN 을 낸다.
결과는 `SWEEP["2x32"]` 에 저장되어 §7 비교표에 쓰인다.

In [10]:
if "2x32" in ARCHS:
    SWEEP["2x32"] = run_eval("2x32")
else:
    print("models_2x32/ 없음 — 건너뜀")

##############################################################################################################
### ARCH = 2x32   (models_root = c:\test\models_2x32)
### ⚠️ 재보정 안 된 서비스: ['auth', 'frontend'] — 학습 시점 임계값 유지
##############################################################################################################

세션 기반 런타임 평가 시작
attack image 정책: 4 benign + 1 attack / window=5 / stride=1

##############################################################################################################
■ SERVICE: auth
##############################################################################################################
threshold = -60.470480

[benign]
  benign_auth.pcap: raw-egress=109831, kept=109831, sessions=888
  pure benign images = 106282 (실제 benign session 내부에서만 stride=1 생성)

[attack: mixed windows]
  scenario              raw     kept   sess  mixed-img       TP       FN    recall
  -----------------------------------------------------------------------------

## §7. arch 비교

서비스별로 arch 를 세로로 쌓아 TP/FP/TN/FN·Recall·FPR·AUC 를 비교하고,
마지막에 arch 전체 합계를 낸다.

`ROLLBK` 행의 Recall/FPR 은 학습 시점 임계값 기준이라 운영 성능이 아니다.
arch 간 우열은 임계값과 무관한 **AUC** 로 판단하는 게 맞다.

In [11]:
# arch 별 TP/FP/TN/FN 비교 — run_pipeline.ipynb 의 "서비스별 최종 요약"을 크기별로 나란히
import numpy as np

if not SWEEP:
    print("아직 실행된 arch 가 없습니다 (§5 셀들을 먼저 실행하세요)")
else:
    # 1) 서비스별 상세
    for svc in SERVICES:
        print(f"\n{'='*118}\n■ {svc}\n{'='*118}")
        print(f"{'arch':8s}{'재보정':>8s}{'TP':>9s}{'FP':>9s}{'TN':>9s}{'FN':>9s}"
              f"{'Recall':>10s}{'FPR':>9s}{'AUC':>9s}")
        print("-" * 118)
        for arch in ARCHS:
            r = SWEEP.get(arch, {}).get(svc)
            if not r:
                print(f"{arch:8s}{'미실행':>8s}"); continue
            m = r["metrics"]
            st = "OK" if ARCH_STATUS[arch][svc]["recalibrated"] else "ROLLBK"
            print(f"{arch:8s}{st:>8s}{m['TP']:9d}{m['FP']:9d}{m['TN']:9d}{m['FN']:9d}"
                  f"{m['recall']*100:9.2f}%{m['fpr']*100:8.2f}%{m['auc']:9.4f}")

    # 2) arch 전체 합계
    print(f"\n{'='*118}\n■ arch 전체 (4개 서비스 합계)\n{'='*118}")
    print(f"{'arch':8s}{'재보정OK':>10s}{'TP':>10s}{'FP':>10s}{'TN':>10s}{'FN':>10s}"
          f"{'Recall':>10s}{'FPR':>9s}{'meanAUC':>10s}{'worstAUC':>10s}")
    print("-" * 118)
    for arch in ARCHS:
        res = SWEEP.get(arch)
        if not res:
            print(f"{arch:8s}{'미실행':>10s}"); continue
        TP = sum(r["metrics"]["TP"] for r in res.values())
        FP = sum(r["metrics"]["FP"] for r in res.values())
        TN = sum(r["metrics"]["TN"] for r in res.values())
        FN = sum(r["metrics"]["FN"] for r in res.values())
        aucs = [r["metrics"]["auc"] for r in res.values()]
        n_ok = sum(1 for s in SERVICES if ARCH_STATUS[arch][s]["recalibrated"])
        rec = TP / (TP + FN) if (TP + FN) else float("nan")
        fpr = FP / (FP + TN) if (FP + TN) else float("nan")
        print(f"{arch:8s}{n_ok:>7d}/{len(SERVICES):<2d}{TP:10d}{FP:10d}{TN:10d}{FN:10d}"
              f"{rec*100:9.2f}%{fpr*100:8.2f}%{np.mean(aucs):10.4f}{min(aucs):10.4f}")



■ auth
arch         재보정       TP       FP       TN       FN    Recall      FPR      AUC
----------------------------------------------------------------------------------------------------------------------
1x8       ROLLBK     5588      911   105371    20161    21.70%    0.86%   0.7613
2x8           OK    22903     1064   105218     2846    88.95%    1.00%   0.9702
1x16      ROLLBK    14418      558   105724    11331    55.99%    0.53%   0.8130
2x16          OK    23584     1860   104422     2165    91.59%    1.75%   0.9800
2x32      ROLLBK    23235     1998   104284     2514    90.24%    1.88%   0.9808

■ post
arch         재보정       TP       FP       TN       FN    Recall      FPR      AUC
----------------------------------------------------------------------------------------------------------------------
1x8           OK     6711        0     1376    13126    33.83%    0.00%   0.7105
2x8           OK    19671        7     1369      166    99.16%    0.51%   0.9982
1x16          OK 